# 🔥 DWSIM - Optimize Natural Gas Processing

> **Created by:**
> Prof. Roymel Rodríguez Carpio, Ph.D.
> Prof. Nicolas Spogis, Ph.D.
> Gabriel Freitas

---

## Import the Libraries

### Libraries for array and dataset manipulation

In [1]:
import numpy as np
import pandas as pd
import time

### Libraries for plotting results

In [2]:
import matplotlib.pyplot as plt
import seaborn as sns


### Set NumPy Seed

In [3]:
seed = 42
np.random.seed(seed)

## Import the Specific Libraries to run DWSIM

In [4]:
# remove the following two lines to run on linux
import pythoncom  # Component of the pywin32 library used to interact with COM (Component Object Model)

pythoncom.CoInitialize()

In [5]:
import clr  # Pythonnet

## Change your DWSIM installation path and load dlls

In [6]:
dwsimpath = r"C:\Users\nicol\AppData\Local\DWSIM"

In [7]:
clr.AddReference(dwsimpath + "\\CapeOpen.dll")
clr.AddReference(dwsimpath + "\\DWSIM.Automation.dll")
clr.AddReference(dwsimpath + "\\DWSIM.Interfaces.dll")
clr.AddReference(dwsimpath + "\\DWSIM.GlobalSettings.dll")
clr.AddReference(dwsimpath + "\\DWSIM.SharedClasses.dll")
clr.AddReference(dwsimpath + "\\DWSIM.Thermodynamics.dll")
clr.AddReference(dwsimpath + "\\DWSIM.UnitOperations.dll")
clr.AddReference(dwsimpath + "\\DWSIM.Inspector.dll")
clr.AddReference(dwsimpath + "\\System.Buffers.dll")
clr.AddReference(dwsimpath + "\\DWSIM.Thermodynamics.ThermoC.dll")



## Open DWSIM Automation def

In [8]:
def open_DWSIM(FlowsheetFile):
    from DWSIM.Automation import Automation3
    Manager = Automation3()
    Flowsheet = Manager.LoadFlowsheet(FlowsheetFile)
    return Manager, Flowsheet

## Read and Save Snapshot defs

In [9]:
def SaveSnapshot(dwsimpath, Flowsheet):
    clr.AddReference(dwsimpath + "\\DWSIM.Interfaces.dll")
    from DWSIM.Interfaces.Enums import SnapshotType
    Type = SnapshotType.ObjectData
    snap = Flowsheet.GetSnapshot(Type)
    return snap


def ReadSnapshot(dwsimpath, snap, Flowsheet):
    clr.AddReference(dwsimpath + "\\DWSIM.Interfaces.dll")
    from DWSIM.Interfaces.Enums import SnapshotType
    Type = SnapshotType.ObjectData
    Flowsheet.RestoreSnapshot(snap, Type)
    return snap

## Open DWSIM File and load initial Snapshot

In [10]:
FlowsheetFileName = "DWSIM/NaturalGasProcessing.dwxmz"
Manager, Flowsheet = open_DWSIM(FlowsheetFileName)

In [11]:
Snapshot = SaveSnapshot(dwsimpath, Flowsheet)

## Call DWSIM -> Change Input Parameters -> Solve -> Get Output Parameters

In [12]:
def run_dwsim_simulation(u):
    global Manager, Flowsheet, Snapshot, dwsimpath

    # Read Snapshot
    ReadSnapshot(dwsimpath, Snapshot, Flowsheet)

    # Extract decision variables
    V_123102_Temp = u[0]
    T_123701_Reboiler = u[1]
    T_123702_Condenser = u[2]
    T_123702_Reboiler = u[3]
    T_123703_Reboiler = u[4]

    # Set Spreadsheet Object
    mySpreadsheet = Flowsheet.GetSpreadsheetObject()
    ws = mySpreadsheet.Worksheets["MAIN"]

    try:
        # Set Input Parameters
        ws.Cells["B2"].Data = V_123102_Temp
        ws.Cells["B3"].Data = T_123701_Reboiler
        ws.Cells["B4"].Data = T_123702_Condenser
        ws.Cells["B5"].Data = T_123702_Reboiler
        ws.Cells["B6"].Data = T_123703_Reboiler

        # Update Input Spreadsheet
        ws.Recalculate()

        # Request a calculation
        Manager.CalculateFlowsheet4(Flowsheet)

        # Check if the Flowsheet was Calculated Correctly
        if Flowsheet.Solved == False:
            # Penalty for failed simulation
            return np.array([1e6, 1e6, 1e6, 1e6, 1e6])

        # Update Output Spreadsheet
        ws.Recalculate()

        # Get Output Parameters
        Sale_Price = float(ws.Cells["B17"].Data)
        GVC1 = float(ws.Cells["B10"].Data)
        GVC2 = float(ws.Cells["B12"].Data)
        GVC5 = float(ws.Cells["B15"].Data)
        GVC5_PVR = float(ws.Cells["B16"].Data)

        # ========================================
        # Objective Function: Maximize Sales Price
        # ========================================
        f_obj = -Sale_Price  # we want to maximize this

        # ========================================
        # Constraints
        # ========================================
        g1 = GVC1  # Should be >= 80 %
        g2 = GVC2  # Should be <=12 %
        g3 = GVC5  # Should be <=2 %
        g4 = GVC5_PVR  # Should be <= 76 kPa

        return np.array([f_obj, g1, g2, g3, g4])

    except Exception as e:
        print(f"Error in simulation: {e}")
        return np.array([1e6, 1e6, 1e6, 1e6, 1e6])



## Variable bounds

In [13]:
# Decision variable bounds
# [V_123102_Temp, T_123701_Reboiler, T_123702_Condenser, T_123702_Reboiler, T_123703_Reboiler]
LB = np.array([-33.0, 60.0, 0.7,  9.0, 45.0])  # Lower bounds
UB = np.array([-17.0, 77.0, 4.5, 27.0, 64.0])  # Upper bounds

bounds = list(zip(LB, UB))

print("\nDecision Variable Bounds:")
print(f"  V_123102_Temp: [{LB[0]:.0f}, {UB[0]:.0f}]")
print(f"  T_123701_Reboiler: [{LB[1]:.0f}, {UB[1]:.0f}]")
print(f"  T_123702_Condenser: [{LB[2]:.0f}, {UB[2]:.0f}]")
print(f"  T_123702_Reboiler: [{LB[3]:.0f}, {UB[3]:.0f}]")
print(f"  T_123703_Reboiler: [{LB[4]:.0f}, {UB[4]:.0f}]")



Decision Variable Bounds:
  V_123102_Temp: [-33, -17]
  T_123701_Reboiler: [60, 77]
  T_123702_Condenser: [1, 4]
  T_123702_Reboiler: [9, 27]
  T_123703_Reboiler: [45, 64]


## Test simulation with initial values

In [14]:
print("=" * 60)
print("TESTING DWSIM SIMULATION - Lower bounds")
print("=" * 60)

f_obj, g1, g2, g3, g4 = run_dwsim_simulation(LB)

print(f"\nTest Input:")
print(f"  V_123102_Temp = {LB[0]:.2f}")
print(f"  T_123701_Reboiler = {LB[1]:.2f}")
print(f"  T_123702_Condenser = {LB[2]:.2f}")
print(f"  T_123702_Reboiler = {LB[3]:.2f}")
print(f"  T_123703_Reboiler = {LB[4]:.2f}")

print(f"\nTest Output:")
print(f"  Objective (Sales Price) = {-f_obj:.2f} US$")
print(f"  g1 (GV.C1 [%]) = {g1:.2f} (should be >= 80 %)")
print(f"  g2 (GLP.C2 [%]) = {g2:.2f} (should be <=12 %)")
print(f"  g3 (GLP.C5 [%]) = {g3:.2f} (should be <=2 %)")
print(f"  g4 (C5+.PVR [kPa]) = {g4:.2f} (should be <= 76 kPa)")

TESTING DWSIM SIMULATION - Lower bounds

Test Input:
  V_123102_Temp = -33.00
  T_123701_Reboiler = 60.00
  T_123702_Condenser = 0.70
  T_123702_Reboiler = 9.00
  T_123703_Reboiler = 45.00

Test Output:
  Objective (Sales Price) = 658.72 US$
  g1 (GV.C1 [%]) = 83.54 (should be >= 80 %)
  g2 (GLP.C2 [%]) = 8.08 (should be <=12 %)
  g3 (GLP.C5 [%]) = 6.71 (should be <=2 %)
  g4 (C5+.PVR [kPa]) = 32.65 (should be <= 76 kPa)


In [15]:
print("=" * 60)
print("TESTING DWSIM SIMULATION - Upper bounds")
print("=" * 60)

f_obj, g1, g2, g3, g4 = run_dwsim_simulation(UB)

print(f"\nTest Input:")
print(f"  V_123102_Temp = {UB[0]:.2f}")
print(f"  T_123701_Reboiler = {UB[1]:.2f}")
print(f"  T_123702_Condenser = {UB[2]:.2f}")
print(f"  T_123702_Reboiler = {UB[3]:.2f}")
print(f"  T_123703_Reboiler = {UB[4]:.2f}")

print(f"\nTest Output:")
print(f"  Objective (Sales Price) = {-f_obj:.2f} US$")
print(f"  g1 (GV.C1 [%]) = {g1:.2f} (should be >= 80 %)")
print(f"  g2 (GLP.C2 [%]) = {g2:.2f} (should be <=12 %)")
print(f"  g3 (GLP.C5 [%]) = {g3:.2f} (should be <=2 %)")
print(f"  g4 (C5+.PVR [kPa]) = {g4:.2f} (should be <= 76 kPa)")

TESTING DWSIM SIMULATION - Upper bounds

Test Input:
  V_123102_Temp = -17.00
  T_123701_Reboiler = 77.00
  T_123702_Condenser = 4.50
  T_123702_Reboiler = 27.00
  T_123703_Reboiler = 64.00

Test Output:
  Objective (Sales Price) = 626.16 US$
  g1 (GV.C1 [%]) = 81.13 (should be >= 80 %)
  g2 (GLP.C2 [%]) = 14.28 (should be <=12 %)
  g3 (GLP.C5 [%]) = 0.00 (should be <=2 %)
  g4 (C5+.PVR [kPa]) = 117.01 (should be <= 76 kPa)


## Class Evaluator to avoid recomputation

In [16]:
class Evaluator:
    def __init__(self):
        self.cache = {}
        self.n_evaluations = 0

    def evaluate(self, x):
        key = tuple(np.round(x, 6))  # Less precision for caching
        if key not in self.cache:
            self.cache[key] = run_dwsim_simulation(x)
            self.n_evaluations += 1
        return self.cache[key]

    def objective(self, x):
        return self.evaluate(x)[0]

    def g1(self, x):
        return self.evaluate(x)[1]

    def g2(self, x):
        return self.evaluate(x)[2]

    def g3(self, x):
        return self.evaluate(x)[3]

    def g4(self, x):
        return self.evaluate(x)[4]

    def reset_counter(self):
        self.n_evaluations = 0

## Create global evaluator

In [17]:
ev = Evaluator()

## Constraint checking function

In [18]:
def check_constraints_scipy(x, constraints, tol=1e-6):
    """Check constraints for scipy optimization"""
    report = []

    for i, c in enumerate(constraints, 1):
        val = np.atleast_1d(c.fun(x))
        lb = np.atleast_1d(c.lb)
        ub = np.atleast_1d(c.ub)

        viol_low = np.maximum(lb - val, 0.0)
        viol_high = np.maximum(val - ub, 0.0)
        violation = np.maximum(viol_low, viol_high)

        ok = np.all(violation <= tol)

        report.append({
            "id": i,
            "ok": ok,
            "value": val,
            "lb": lb,
            "ub": ub,
            "violation": violation
        })

    return report

# ========================================
# Create LHS Dataset
# ========================================

In [19]:
from scipy.stats import qmc, norm
from tqdm import tqdm

In [20]:
# Bounds array: each row is [min, max] for each variable
LHS_bounds = np.column_stack((LB, UB))

In [21]:
n_samples = 2000
n_variables = LHS_bounds.shape[0]

In [22]:
# =============================================================================
# Generate LHS samples
# =============================================================================
print("Generating Latin Hypercube Samples...")

# Create LHS sampler with seed for reproducibility
sampler = qmc.LatinHypercube(d=n_variables, seed=42)

# Generate samples in [0, 1] range
lhs_samples_unit = sampler.random(n=n_samples)

# Scale samples to actual bounds
lhs_samples = qmc.scale(lhs_samples_unit, LHS_bounds[:, 0], LHS_bounds[:, 1])

print(f"Generated {n_samples} LHS samples successfully!")

Generating Latin Hypercube Samples...
Generated 2000 LHS samples successfully!


In [23]:
# =============================================================================
# Run DWSIM simulations for all samples
# =============================================================================
print(f"\nRunning DWSIM simulations for {n_samples} samples...")
print("This may take a while...\n")

# Initialize output arrays
outputs = np.zeros((n_samples, 5))

# Run simulations with progress bar
for i, sample in enumerate(tqdm(lhs_samples, desc="Simulating", unit="sample")):
    outputs[i, :] = run_dwsim_simulation(sample)

print("\nSimulations completed!")


Running DWSIM simulations for 2000 samples...
This may take a while...



Simulating: 100%|██████████| 2000/2000 [2:17:30<00:00,  4.13s/sample]  


Simulations completed!


In [24]:
# =============================================================================
# Create DataFrame and save dataset
# =============================================================================
# Column names
input_columns = ['V_123102_Temp',
                 'T_123701_Reboiler',
                 'T_123702_Condenser',
                 'T_123702_Reboiler',
                 'T_123703_Reboiler']

output_columns = ['Sales_Price', 'GVC1', 'GVC2', 'GVC5', 'GVC5_PVR']

# Create DataFrame
df = pd.DataFrame(
    data=np.hstack([lhs_samples, outputs]),
    columns=input_columns + output_columns
)

# Count valid and invalid samples
mask_invalid = (df[output_columns] == 1e6).all(axis=1)
df_clean = df[~mask_invalid]
n_valid = df_clean.shape[0]
n_invalid = n_samples - n_valid

print(f"\n{'='*60}")
print(f"Dataset Summary")
print(f"{'='*60}")
print(f"Total samples:   {n_samples}")
print(f"Valid samples:   {n_valid} ({100*n_valid/n_samples:.1f}%)")
print(f"Invalid samples: {n_invalid} ({100*n_invalid/n_samples:.1f}%)")
print(f"{'='*60}")


Dataset Summary
Total samples:   2000
Valid samples:   2000 (100.0%)
Invalid samples: 0 (0.0%)


In [25]:
# Display statistics
print("\nDataset Statistics:")
print(df.describe().round(4))


Dataset Statistics:
       V_123102_Temp  T_123701_Reboiler  T_123702_Condenser  \
count      2000.0000          2000.0000           2000.0000   
mean        -24.9999            68.5000              2.6000   
std           4.6200             4.9087              1.0973   
min         -33.0000            60.0038              0.7011   
25%         -28.9971            64.2485              1.6501   
50%         -24.9984            68.5027              2.6007   
75%         -21.0046            72.7450              3.5500   
max         -17.0031            76.9983              4.4994   

       T_123702_Reboiler  T_123703_Reboiler  Sales_Price       GVC1  \
count          2000.0000          2000.0000    2000.0000  2000.0000   
mean             18.0000            54.5000    -644.5029    82.4302   
std               5.1974             5.4862      13.8967     0.9233   
min               9.0069            45.0044    -673.3515    80.7304   
25%              13.5050            49.7551    -656.4881

In [26]:
# =============================================================================
# Save dataset to CSV
# =============================================================================
# Save full dataset
output_file_full = "datasets/dataset_full.csv"
df.to_csv(output_file_full, index=False)
print(f"\nFull dataset saved to: {output_file_full}")

# Save clean dataset
output_file_clean = "datasets/dataset_clean.csv"
df_clean.to_csv(output_file_clean, index=False)
print(f"Clean dataset saved to: {output_file_clean}")

print(f"\n{'='*60}")
print("Dataset generation completed successfully!")
print(f"{'='*60}")


Full dataset saved to: dataset_full.csv
Clean dataset saved to: dataset_clean.csv

Dataset generation completed successfully!


# ========================================
# LOCAL OPTIMIZATION: SLSQP (Scipy - Baseline)
# ========================================

In [34]:
# ============================================================
# Load LHS dataset
# ============================================================
df = pd.read_csv("datasets/dataset_clean.csv")  # adjust path as needed

# Column names (adjust if yours differ)
decision_vars = [
    'V_123102_Temp',
    'T_123701_Reboiler',
    'T_123702_Condenser',
    'T_123702_Reboiler',
    'T_123703_Reboiler',
]
obj_col = 'Sales_Price'
g1_col = 'GVC1'       # >= 80
g2_col = 'GVC2'       # <= 12
g3_col = 'GVC5'       # <= 2
g4_col = 'GVC5_PVR'   # <= 76

# ============================================================
# Define constraint limits
# (adjust these to match your actual NonlinearConstraint bounds)
# ============================================================
constraints_limits = {
    g1_col: {'lb': 80.0, 'ub': np.inf},     # g1 >= 80
    g2_col: {'lb': 0.0, 'ub': 12.0},        # g2 <= 12
    g3_col: {'lb': 0.0, 'ub': 2.0},         # g3 <= 2
    g4_col: {'lb': 0.0, 'ub': 76.0},        # g4 <= 76
}

# ============================================================
# Check feasibility
# ============================================================
feasible_mask = pd.Series(True, index=df.index)

for col, lim in constraints_limits.items():
    if np.isfinite(lim['lb']):
        feasible_mask &= (df[col] >= lim['lb'])
    if np.isfinite(lim['ub']):
        feasible_mask &= (df[col] <= lim['ub'])

df_feasible = df[feasible_mask]
n_total = len(df)
n_feasible = len(df_feasible)

print(f"Total LHS samples: {n_total}")
print(f"Feasible samples:  {n_feasible} ({100*n_feasible/n_total:.1f}%)")

# ============================================================
# Select best x0
# ============================================================
if n_feasible > 0:
    # Best feasible point (min Sales_Price = most negative = highest real price)
    best_idx = df_feasible[obj_col].idxmin()
    best_row = df_feasible.loc[best_idx]
    print(f"\n--- Best feasible point (index {best_idx}) ---")
else:
    # No feasible point: pick the one with minimum total violation
    print("\nNo fully feasible point found. Selecting least-violated point...")

    violation = pd.Series(0.0, index=df.index)
    for col, lim in constraints_limits.items():
        if np.isfinite(lim['lb']):
            violation += np.maximum(lim['lb'] - df[col], 0.0)
        if np.isfinite(lim['ub']):
            violation += np.maximum(df[col] - lim['ub'], 0.0)

    best_idx = violation.idxmin()
    best_row = df.loc[best_idx]
    print(f"\n--- Least-violated point (index {best_idx}, "
          f"total violation = {violation[best_idx]:.4f}) ---")

# Extract x0
x0 = best_row[decision_vars].values.astype(float)

print(f"\nObjective (Sales_Price): {best_row[obj_col]:.4f}")
print(f"Real Price: {-best_row[obj_col]:.4f}")
print(f"\nDecision variables (x0):")
for name, val in zip(decision_vars, x0):
    print(f"  {name} = {val:.4f}")

print(f"\nConstraint values:")
for col, lim in constraints_limits.items():
    val = best_row[col]
    lb_str = f"{lim['lb']:.1f}" if np.isfinite(lim['lb']) else "-inf"
    ub_str = f"{lim['ub']:.1f}" if np.isfinite(lim['ub']) else "inf"
    ok = True
    if np.isfinite(lim['lb']) and val < lim['lb']:
        ok = False
    if np.isfinite(lim['ub']) and val > lim['ub']:
        ok = False
    status = "✓" if ok else "✗"
    print(f"  {col} = {val:.4f}  [{lb_str}, {ub_str}]  {status}")

# ============================================================
# x0 is ready to use in your optimization
# ============================================================
print(f"\nx0 = np.array({list(x0)})")

Total LHS samples: 2000
Feasible samples:  281 (14.1%)

--- Best feasible point (index 1591) ---

Objective (Sales_Price): -662.8284
Real Price: 662.8284

Decision variables (x0):
  V_123102_Temp = -32.8294
  T_123701_Reboiler = 64.0769
  T_123702_Condenser = 4.2981
  T_123702_Reboiler = 13.8861
  T_123703_Reboiler = 47.4421

Constraint values:
  GVC1 = 83.7331  [80.0, inf]  ✓
  GVC2 = 11.9135  [0.0, 12.0]  ✓
  GVC5 = 0.1878  [0.0, 2.0]  ✓
  GVC5_PVR = 53.5710  [0.0, 76.0]  ✓

x0 = np.array([-32.82942291491444, 64.07694006124437, 4.298112932449205, 13.886056746901064, 47.442059795382654])


In [36]:
from scipy.optimize import minimize, NonlinearConstraint

print("\n\n" + "=" * 60)
print("OPTIMIZATION 1: SLSQP (SciPy - Local Optimizer)")
print("=" * 60)

# Reset evaluator
ev.reset_counter()
ev.cache.clear()

# Constraints for SLSQP
constraints_scipy = [
    NonlinearConstraint(ev.g1, 80.0, np.inf),       # should be >= 80
    NonlinearConstraint(ev.g2, 0.0, 12.0),          # should be <=12 %
    NonlinearConstraint(ev.g3, 0.0, 2.0),           # should be <=2 %
    NonlinearConstraint(ev.g4, 0.0, 76.0),          # should be <= 76 kPa
]

# Callback
n_iter_slsqp = 0


def callback_slsqp(xk):
    global n_iter_slsqp
    n_iter_slsqp += 1
    print(f"Iter {n_iter_slsqp} | Evaluations {ev.n_evaluations}")


# Options
method = 'SLSQP'
options = {
    'maxiter': 100,
    'ftol': 1e-2,
    'disp': True,
    'eps': 0.5
}

# Run optimization
start_time = time.time()
res_slsqp = minimize(
    ev.objective,
    x0,
    method=method,
    bounds=bounds,
    constraints=constraints_scipy,
    options=options,
    callback=callback_slsqp
)
time_slsqp = time.time() - start_time



OPTIMIZATION 1: SLSQP (SciPy - Local Optimizer)
Iter 1 | Evaluations 9
Iter 2 | Evaluations 15
Iter 3 | Evaluations 21
Iter 4 | Evaluations 27
Iter 5 | Evaluations 32
Optimization terminated successfully    (Exit mode 0)
            Current function value: -663.8537088977982
            Iterations: 5
            Function evaluations: 30
            Gradient evaluations: 5


In [37]:
# Results
print("\n" + "=" * 60)
print("SLSQP RESULTS:")
print("=" * 60)
print(f"Converged: {res_slsqp.success}")
print(f"Message: {res_slsqp.message}")
print(f"\nObjective Value:")
print(f"  Sales Price = {-res_slsqp.fun:.2f}")

print(f"\nOptimal Solution:")
print(f"  V_123102_Temp = {res_slsqp.x[0]:.2f}")
print(f"  T_123701_Reboiler = {res_slsqp.x[1]:.2f}")
print(f"  T_123702_Condenser = {res_slsqp.x[2]:.2f}")
print(f"  T_123702_Reboiler = {res_slsqp.x[3]:.2f}")
print(f"  T_123703_Reboiler = {res_slsqp.x[4]:.2f}")


print(f"\nPerformance:")
print(f"  Iterations: {n_iter_slsqp}")
print(f"  DWSIM evaluations: {ev.n_evaluations}")
print(f"  Optimization time: {time_slsqp:.2f} seconds")

# Check constraints
report = check_constraints_scipy(res_slsqp.x, constraints_scipy)
print("\nConstraint Check:")
for r in report:
    status = "✓ OK" if r['ok'] else "✗ VIOLATED"
    print(f"  Constraint {r['id']}: {status}")
    print(f"    value = {r['value'][0]:.4f}, bounds = [{r['lb'][0]:.2f}, {r['ub'][0]:.2f}]")



SLSQP RESULTS:
Converged: True
Message: Optimization terminated successfully

Objective Value:
  Sales Price = 663.85

Optimal Solution:
  V_123102_Temp = -33.00
  T_123701_Reboiler = 63.53
  T_123702_Condenser = 4.31
  T_123702_Reboiler = 15.96
  T_123703_Reboiler = 48.00

Performance:
  Iterations: 5
  DWSIM evaluations: 32
  Optimization time: 253.88 seconds

Constraint Check:
  Constraint 1: ✓ OK
    value = 83.7441, bounds = [80.00, inf]
  Constraint 2: ✓ OK
    value = 11.9999, bounds = [0.00, 12.00]
  Constraint 3: ✓ OK
    value = 0.0056, bounds = [0.00, 2.00]
  Constraint 4: ✗ VIOLATED
    value = 76.0021, bounds = [0.00, 76.00]
